# SEGMENT 2 — Feature Selection & Preprocessing Pipelines

**Project**: HDB Resale Price Prediction  
**Builds on**: Segment 1 decisions  

---

## Segment Objectives
1. Drop non-predictive, redundant, and high-cardinality columns
2. Define final numeric and categorical feature lists
3. Perform 80/20 train/test split (random_state=42)
4. Build **Pipeline A** — with StandardScaler (for LR, KNN, Ridge, Lasso, Elastic Net)
5. Build **Pipeline B** — without scaler (for Decision Tree, Gradient Boosting)
6. Fit pipelines on training data only — no data leakage
7. Transform both train and test sets
8. Save processed arrays for use in downstream segments

---
> **Leakage guard**: Imputer and Scaler are ONLY fit on `X_train`. `X_test` is only transformed.

---
## Step 2.1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Imports complete.')

---
## Step 2.2 — Load Data

In [ ]:
train = pd.read_csv('train.csv', low_memory=False)
test  = pd.read_csv('test.csv',  low_memory=False)

TARGET = 'resale_price'
print(f'Train shape: {train.shape}')
print(f'Test shape : {test.shape}')

---
## Step 2.3 — Drop Non-Predictive & Redundant Columns

In [ ]:
# ── Columns to DROP (from Segment 1 decisions) ───────────────────────────────

DROP_ID_LIKE = [
    'id',
    'block', 'street_name', 'address', 'postal',
    'bus_stop_name', 'mrt_name', 'pri_sch_name', 'sec_sch_name',
]

DROP_REDUNDANT = [
    'floor_area_sqft',       # same info as floor_area_sqm
    'full_flat_type',        # compound of flat_type + flat_model
    'Tranc_YearMonth',       # split into Tranc_Year + Tranc_Month
    'storey_range',          # text version of mid_storey
    'lower', 'upper', 'mid', # redundant with mid_storey
    'lease_commence_date',   # redundant with hdb_age
    'year_completed',        # redundant with hdb_age
]

DROP_SPARSE = [
    'Mall_Within_500m',      # 62% missing
    'Hawker_Within_500m',    # 65% missing
]

DROP_REDUNDANT_GEO = [
    'mrt_latitude', 'mrt_longitude',
    'bus_stop_latitude', 'bus_stop_longitude',
    'pri_sch_latitude', 'pri_sch_longitude',
    'sec_sch_latitude', 'sec_sch_longitude',
]

ALL_DROP = DROP_ID_LIKE + DROP_REDUNDANT + DROP_SPARSE + DROP_REDUNDANT_GEO
# Only drop columns that actually exist
ALL_DROP_TRAIN = [c for c in ALL_DROP if c in train.columns]
ALL_DROP_TEST  = [c for c in ALL_DROP if c in test.columns]

train_clean = train.drop(columns=ALL_DROP_TRAIN)
test_clean  = test.drop(columns=ALL_DROP_TEST)

print(f'After drop: train {train_clean.shape}, test {test_clean.shape}')
print(f'Dropped {len(ALL_DROP_TRAIN)} columns from train')

---
## Step 2.4 — Define Final Feature Lists

In [ ]:
# ── Manually reviewed final feature lists ────────────────────────────────────

NUMERIC_FEATURES = [
    # Transactional
    'Tranc_Year', 'Tranc_Month',
    # Size
    'floor_area_sqm',
    # Height
    'mid_storey', 'max_floor_lvl',
    # Age
    'hdb_age',
    # Availability
    'total_dwelling_units', 'vacancy',
    # Past transactions by flat type
    '1room_sold', '2room_sold', '3room_sold', '4room_sold',
    '5room_sold', 'exec_sold', 'multigen_sold', 'studio_apartment_sold',
    # Rental
    '1room_rental', '2room_rental', '3room_rental', 'other_room_rental',
    # Amenities — Recreational
    'Mall_Nearest_Distance', 'Mall_Within_1km', 'Mall_Within_2km',
    # Amenities — Food
    'Hawker_Nearest_Distance', 'Hawker_Within_1km', 'Hawker_Within_2km',
    'hawker_food_stalls', 'hawker_market_stalls',
    # Transport
    'mrt_nearest_distance', 'bus_interchange', 'mrt_interchange',
    'bus_stop_nearest_distance',
    # Education
    'pri_sch_nearest_distance', 'pri_sch_affiliation',
    'sec_sch_nearest_dist', 'cutoff_point', 'affiliation',
    # Geography
    'Latitude', 'Longitude',
]

CATEGORICAL_FEATURES = [
    'town', 'planning_area',
    'flat_type', 'flat_model',
    'residential', 'commercial',
    'market_hawker', 'multistorey_carpark', 'precinct_pavilion',
]

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Validate all features exist
missing_in_train = [f for f in ALL_FEATURES if f not in train_clean.columns]
if missing_in_train:
    print(f'WARNING — features not found in train: {missing_in_train}')
else:
    print(f'All {len(ALL_FEATURES)} features confirmed in train.')
    print(f'  Numeric    : {len(NUMERIC_FEATURES)}')
    print(f'  Categorical: {len(CATEGORICAL_FEATURES)}')

---
## Step 2.5 — Train/Test Split (80/20, random_state=42)

In [ ]:
X = train_clean[ALL_FEATURES]
y = train_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train : {y_train.shape},  mean=${y_train.mean():,.0f}')
print(f'y_test  : {y_test.shape},   mean=${y_test.mean():,.0f}')

---
## Step 2.6 — Build Pipeline A (Scaled — for LR, KNN, Ridge, Lasso, Elastic Net)

**Numeric branch**: `SimpleImputer(median)` → `StandardScaler()`  
**Categorical branch**: `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown='ignore')`

In [ ]:
# ── Pipeline A — with scaling ────────────────────────────────────────────────
numeric_pipeline_A = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

categorical_pipeline_A = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor_A = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline_A,    NUMERIC_FEATURES),
        ('cat', categorical_pipeline_A, CATEGORICAL_FEATURES),
    ],
    remainder='drop'
)

# Fit on TRAIN only — no leakage
X_train_A = preprocessor_A.fit_transform(X_train)
X_test_A  = preprocessor_A.transform(X_test)

print(f'Pipeline A — X_train_A shape: {X_train_A.shape}')
print(f'Pipeline A — X_test_A  shape: {X_test_A.shape}')
print(f'(Columns expanded from {len(ALL_FEATURES)} to {X_train_A.shape[1]} due to OHE)')

---
## Step 2.7 — Build Pipeline B (Tree-Safe — for Decision Tree, Gradient Boosting)

**Numeric branch**: `SimpleImputer(median)` only — NO scaler  
**Categorical branch**: `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown='ignore')`

> Tree models split on threshold values — scaling does not affect the split logic. Applying StandardScaler to tree-based models is unnecessary and slightly misleading.

In [ ]:
# ── Pipeline B — no scaling ──────────────────────────────────────────────────
numeric_pipeline_B = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    # No StandardScaler for tree models
])

categorical_pipeline_B = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor_B = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline_B,    NUMERIC_FEATURES),
        ('cat', categorical_pipeline_B, CATEGORICAL_FEATURES),
    ],
    remainder='drop'
)

# Fit on TRAIN only
X_train_B = preprocessor_B.fit_transform(X_train)
X_test_B  = preprocessor_B.transform(X_test)

print(f'Pipeline B — X_train_B shape: {X_train_B.shape}')
print(f'Pipeline B — X_test_B  shape: {X_test_B.shape}')

---
## Step 2.8 — Retrieve Feature Names After OHE

In [ ]:
def get_feature_names(preprocessor, numeric_features, categorical_features):
    cat_encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']
    cat_names   = cat_encoder.get_feature_names_out(categorical_features).tolist()
    return numeric_features + cat_names

feature_names_A = get_feature_names(preprocessor_A, NUMERIC_FEATURES, CATEGORICAL_FEATURES)
feature_names_B = get_feature_names(preprocessor_B, NUMERIC_FEATURES, CATEGORICAL_FEATURES)

print(f'Total features after encoding (Pipeline A): {len(feature_names_A)}')
print(f'Total features after encoding (Pipeline B): {len(feature_names_B)}')
print()
print('OHE-generated features sample (first 20 cat features):')
print(feature_names_A[len(NUMERIC_FEATURES):len(NUMERIC_FEATURES)+20])

---
## Step 2.9 — Save Preprocessors and Arrays for Downstream Use

In [ ]:
os.makedirs('artifacts', exist_ok=True)

# Save preprocessors
joblib.dump(preprocessor_A, 'artifacts/preprocessor_A.pkl')
joblib.dump(preprocessor_B, 'artifacts/preprocessor_B.pkl')

# Save feature name lists
joblib.dump(feature_names_A, 'artifacts/feature_names_A.pkl')
joblib.dump(feature_names_B, 'artifacts/feature_names_B.pkl')

# Save processed arrays
np.save('artifacts/X_train_A.npy', X_train_A)
np.save('artifacts/X_test_A.npy',  X_test_A)
np.save('artifacts/X_train_B.npy', X_train_B)
np.save('artifacts/X_test_B.npy',  X_test_B)
np.save('artifacts/y_train.npy',   y_train.values)
np.save('artifacts/y_test.npy',    y_test.values)

print('Saved to artifacts/:')
for f in sorted(os.listdir('artifacts')):
    size = os.path.getsize(f'artifacts/{f}') / 1024
    print(f'  {f:<40} {size:.1f} KB')

---
## Step 2.10 — Preprocessing Validation Checks

In [ ]:
# ── Check 1: No NaN remaining after preprocessing ────────────────────────────
print('NaN check after Pipeline A:')
print(f'  X_train_A: {np.isnan(X_train_A).sum()} NaNs')
print(f'  X_test_A : {np.isnan(X_test_A).sum()} NaNs')

print('NaN check after Pipeline B:')
print(f'  X_train_B: {np.isnan(X_train_B).sum()} NaNs')
print(f'  X_test_B : {np.isnan(X_test_B).sum()} NaNs')

# ── Check 2: Scaling verification for Pipeline A ─────────────────────────────
print()
print('Pipeline A numeric column stats (should be ~mean=0, std=1 for scaled cols):')
means = X_train_A[:, :len(NUMERIC_FEATURES)].mean(axis=0)
stds  = X_train_A[:, :len(NUMERIC_FEATURES)].std(axis=0)
scale_check = pd.DataFrame({'Feature': NUMERIC_FEATURES[:5], 'Mean': means[:5], 'Std': stds[:5]})
print(scale_check.to_string(index=False))

# ── Check 3: No target leakage ────────────────────────────────────────────────
print()
print('Leakage check: resale_price NOT in feature list.')
assert 'resale_price' not in feature_names_A, 'LEAKAGE DETECTED in Pipeline A!'
assert 'resale_price' not in feature_names_B, 'LEAKAGE DETECTED in Pipeline B!'
print('  PASS — no target leakage detected.')

---
## Step 2.11 — Preprocessing Summary Table

In [ ]:
summary = pd.DataFrame([
    ['Pipeline A', 'LR, KNN, Ridge, Lasso, Elastic Net', 'Median impute + StandardScaler', 'most_frequent + OHE', X_train_A.shape[1]],
    ['Pipeline B', 'Decision Tree, Gradient Boosting',   'Median impute only',             'most_frequent + OHE', X_train_B.shape[1]],
], columns=['Pipeline', 'Used By', 'Numeric Treatment', 'Categorical Treatment', 'Output Features'])

print(summary.to_string(index=False))

---
## ✅ Segment 2 Complete

**Outputs**:
- `artifacts/preprocessor_A.pkl` — fit preprocessor (scaled)
- `artifacts/preprocessor_B.pkl` — fit preprocessor (tree-safe)
- `artifacts/X_train_A.npy`, `X_test_A.npy`, `X_train_B.npy`, `X_test_B.npy`
- `artifacts/y_train.npy`, `y_test.npy`

**Next**: Proceed to `03_baseline_models.ipynb`
- Train Linear Regression (Pipeline A)
- Train KNN Regressor (Pipeline A)
- Train Decision Tree Regressor (Pipeline B)
- Compare MAE, RMSE, R² on test set
- Interpret bias-variance behavior